# Strings em C

*Rode uma célula com ▶ (ou Shift+Enter): o Sphinx compila e roda a célula no seu computador. Mude o código e rode de novo: este notebook é a sua cópia.*

**Depois desta unidade você consegue:**
- Guardar texto em arrays de char terminados por '\0', com espaço para o terminador, e ler linhas inteiras com segurança com fgets.
- Usar strlen, strcmp, strcpy, strchr e snprintf, e explicar por que == não compara strings.
- Percorrer uma string caractere por caractere para dividir, transformar ou quebrar o texto.

**Neste notebook**, uma ideia de cada vez (rode cada célula antes de seguir):
1. Uma string é um array de char que termina em '\0'
2. Espaço para o '\0'
3. As ferramentas de <string.h>
4. Mudando caracteres
5. Lendo uma linha com segurança
6. Números para texto e de volta

Os **exercícios** deste notebook terminam com linhas `check(...)`: testes mínimos que imprimem ✓ quando o seu código está certo. Rode a próxima célula uma vez antes, para eles poderem usar o `check`.

In [ ]:
// shared: check_int e check_str imprimem ✓ ou ✗; os exercícios abaixo usam
void check_int(const char *what, long actual, long expected) {
    if (actual == expected) {
        printf("✓ %s\n", what);
    } else {
        printf("✗ %s: esperado %ld, veio %ld\n", what, expected, actual);
    }
}

void check_str(const char *what, const char *actual, const char *expected) {
    if (actual == NULL) {
        actual = "(NULL)";
    }
    if (strcmp(actual, expected) == 0) {
        printf("✓ %s\n", what);
    } else {
        printf("✗ %s: esperado \"%s\", veio \"%s\"\n", what, expected, actual);
    }
}

## 1. Uma string é um array de char que termina em '\0'

O C não tem o tipo `String`. Uma string é um array de `char`, e o texto termina no caractere especial `'\0'` (o byte 0). `char s[] = "hi";` ocupa **3** bytes: `'h'`, `'i'` e `'\0'`. Toda função de string acha o fim andando até o `'\0'`, então ele precisa estar sempre lá. Os slides mostram os bytes.

#### 🎞️ Uma string em C, byte a byte

![Uma string em C é um array de char que termina em '\0': strlen conta até ele, e strcat escreve sobre ele e põe um novo no fim](string-bytes-steps.pt-br.svg)

In [ ]:
char s[] = "hi";
printf("sizeof: %zu, strlen: %zu\n", sizeof s, strlen(s));
for (size_t i = 0; i < sizeof s; i++) {
    printf("s[%zu] = '%c' (código %d)\n", i, s[i] ? s[i] : '0', s[i]);
}
printf("%s\n", s);

🧪 **Experimente:** mude o texto para `"hello"`: o que `sizeof` e `strlen` dão agora? Depois ponha `s[1] = '\0';` antes do último `printf`: onde o texto termina?

### ✏️ Exercício: O seu próprio strlen

Conte os chars antes do `'\0'`, sem chamar strlen.

In [ ]:
int my_strlen(const char *s) {
    // TODO: avance até s[i] ser '\0'
    return 0;
}

int main(void) {
    check_int("hello", my_strlen("hello"), 5);
    check_int("vazia", my_strlen(""), 0);
    check_int("com espaço", my_strlen("a b"), 3);
    return 0;
}

<details>
<summary>Ver uma resposta</summary>

```c
int my_strlen(const char *s) {
    int n = 0;
    while (s[n] != '\0') {
        n++;
    }
    return n;
}

int main(void) {
    check_int("hello", my_strlen("hello"), 5);
    check_int("vazia", my_strlen(""), 0);
    check_int("com espaço", my_strlen("a b"), 3);
    return 0;
}
```

</details>

## 2. Espaço para o '\0'

Um array para texto precisa de **uma posição a mais que o texto mais longo**: `char name[32]` cabe 31 caracteres mais o `'\0'`. Escrever depois do fim é comportamento indefinido, como em qualquer array. `snprintf(buffer, sizeof buffer, formato, …)` monta texto como o `printf`, mas nunca escreve mais do que o buffer comporta, sempre pondo o `'\0'`. Ele devolve o tamanho que o texto inteiro precisaria, então dá para saber quando cortou.

In [ ]:
char label[16];
snprintf(label, sizeof label, "Sala %d-%c", 12, 'B');
printf("[%s] usa %zu de %zu bytes\n", label, strlen(label) + 1, sizeof label);

char small[8];
char sentence[32];
strcpy(sentence, "uma frase comprida");  // imagine que foi digitada
int needed = snprintf(small, sizeof small, "%s", sentence);
printf("[%s]: cortada para caber; o texto inteiro precisava de %d chars\n", small, needed);

🧪 **Experimente:** faça `small` ter 4 bytes: quantos caracteres cabem?

## 3. As ferramentas de <string.h>

| Função | Faz |
|---|---|
| `strlen(s)` | o tamanho, sem contar o `'\0'` |
| `strcmp(a, b)` | 0 quando iguais; negativo ou positivo pela ordem do dicionário |
| `strcpy(dest, src)` | copia src em dest (dest precisa ter espaço) |
| `strcat(dest, src)` | acrescenta src no fim de dest (espaço de novo) |
| `strchr(s, c)` | um ponteiro para o primeiro `c` em s, ou `NULL` |

**Nunca compare strings com `==`:** ele compara os endereços dos arrays, não as letras.

In [ ]:
char a[20] = "pera";
char b[20] = "pera";
printf("strcmp(a, b) == 0? %d\n", strcmp(a, b) == 0);
printf("strcmp(\"abacate\", \"pera\") < 0? %d\n", strcmp("abacate", "pera") < 0);
strcat(a, "s");
strcpy(b, "figo");
printf("%s %s (%zu letras)\n", a, b, strlen(a));
char *dash = strchr("2024-11-26", '-');
printf("depois do primeiro traço: %s\n", dash + 1);

🧪 **Experimente:** faça `a` ter só 5 bytes (`char a[5]`) e rode: o `strcat` escreve depois do fim, e a verificação de memória acusa.

## 4. Mudando caracteres

Um array de `char` pode ser mudado caractere por caractere. Ande até o `'\0'` (sem precisar do `strlen`) e mude `s[i]`. Cuidado com as duas formas de escrever texto: `char s[] = "hi";` é o seu próprio array, que você pode mudar; `const char *p = "hi";` aponta para um **literal**, guardado numa memória só de leitura: mudá-lo é comportamento indefinido.

In [ ]:
char word[] = "banana";
for (int i = 0; word[i] != '\0'; i++) {
    if (word[i] == 'a') {
        word[i] = 'o';
    }
}
printf("%s\n", word);
const char *fixed = "texto literal";
printf("%s\n", fixed);

🧪 **Experimente:** tente `fixed[0] = 'T';` e leia o erro: o `const` protege o literal.

### ✏️ Exercício: Maiúsculas no lugar

Troque cada letra de a a z por maiúscula, na própria string. Letras são números: `'a' - 'A'` é a distância entre as duas caixas.

In [ ]:
void to_upper_str(char *s) {
    // TODO: troque cada char minúsculo
}

int main(void) {
    char word[] = "hello";
    to_upper_str(word);
    check_str("hello", word, "HELLO");
    char mixed[] = "Hi, 2 you";
    to_upper_str(mixed);
    check_str("só letras mudam", mixed, "HI, 2 YOU");
    return 0;
}

<details>
<summary>Ver uma resposta</summary>

```c
void to_upper_str(char *s) {
    for (int i = 0; s[i] != '\0'; i++) {
        if (s[i] >= 'a' && s[i] <= 'z') {
            s[i] = (char) (s[i] - 'a' + 'A');
        }
    }
}

int main(void) {
    char word[] = "hello";
    to_upper_str(word);
    check_str("hello", word, "HELLO");
    char mixed[] = "Hi, 2 you";
    to_upper_str(mixed);
    check_str("só letras mudam", mixed, "HI, 2 YOU");
    return 0;
}
```

</details>

## 5. Lendo uma linha com segurança

Nos desafios, leia uma linha com `fgets(buffer, sizeof buffer, stdin)`: ele para na quebra de linha ou quando o buffer enche, então nunca escreve depois do fim. Ele **mantém** o `'\n'`, que você costuma tirar com `buffer[strcspn(buffer, "\n")] = '\0';` (`strcspn` dá o índice do primeiro `'\n'`, ou o tamanho quando não há nenhum). Abaixo, a linha "digitada" está escrita direto.

In [ ]:
char line[32] = "Ada Lovelace\n";       // o que o fgets deixaria
printf("antes:  [%s]\n", line);
line[strcspn(line, "\n")] = '\0';
printf("depois: [%s]\n", line);

🧪 **Experimente:** tire o `\n` do texto inicial e rode: o `strcspn` devolve o tamanho, então nada quebra.

## 6. Números para texto e de volta

`strtol(texto, &fim, 10)` lê um número inteiro de um texto e diz onde parou, então dá para conferir se o texto inteiro era um número (`atoi` é mais simples, mas não informa erros). No sentido contrário, o `snprintf` escreve um número num buffer.

In [ ]:
const char *inputs[] = {"42", "12abc", "abc"};
for (int i = 0; i < 3; i++) {
    char *end;
    long value = strtol(inputs[i], &end, 10);
    if (end == inputs[i]) {
        printf("%-6s: não é número\n", inputs[i]);
    } else if (*end != '\0') {
        printf("%-6s: %ld, e depois o texto extra \"%s\"\n", inputs[i], value, end);
    } else {
        printf("%-6s: %ld\n", inputs[i], value);
    }
}
char buffer[16];
snprintf(buffer, sizeof buffer, "%d", 2026);
printf("%s tem %zu dígitos\n", buffer, strlen(buffer));

🧪 **Experimente:** acrescente `"-7"` e `" 8"` (com espaço) nas entradas. Depois imprima `atoi("12abc")` e `atoi("abc")`: dá para distinguir um erro de um zero?

## Em resumo

O C não tem um tipo `String`: uma string é um array de `char` cujo texto termina com o caractere `'\0'`. As funções acham o fim procurando por ele, então o array precisa de uma posição a mais. `<string.h>` tem as ferramentas: `strlen`, `strcmp`, `strcpy`, `strchr`.

```c
#include <stdio.h>
#include <string.h>

int main(void) {
    char name[32];                              // até 31 caracteres + '\0'
    if (fgets(name, sizeof name, stdin) == NULL) {
        return 1;
    }
    name[strcspn(name, "\n")] = '\0';           // o fgets mantém a quebra de linha
    if (strcmp(name, "Ada") == 0) {             // == compararia endereços
        printf("Hello again, %s!\n", name);
    }
    printf("%zu letters\n", strlen(name));
    return 0;
}
```

O `fgets` para na quebra de linha ou quando o array enche, então nunca escreve depois do fim. `snprintf(buffer, sizeof buffer, …)` monta texto com a mesma garantia.

**Cuidado:** uma string sem `'\0'` continua pela memória que vier depois. Deixe espaço para ele: `"Ada"` precisa de 4 chars.

## Leituras

- [Strings](https://beej.us/guide/bgc/html/split/strings.html) (Beej's Guide to C Programming, chapter 7, 8 min): Literais de string, variáveis de string, o tamanho, o terminador '\0', e por que copiar uma string exige strcpy. "String Variables as Arrays" é a seção principal.
- [Lecture 2: Strings](https://cs50.harvard.edu/x/2026/notes/2/#strings) (CS50x (Harvard), lecture notes, 6 min): Opcional: figuras de uma string na memória, terminando em \0, e "String Length". O tipo string do CS50 é um nome próprio para char *; no Sphinx escreva arrays de char.

## Pratique

O notebook cobriu o básico. Agora aplique nos desafios da unidade, na barra lateral do Sphinx, onde o Submit roda todos os testes, inclusive os ocultos:

- **Estatísticas de palavras** · Fácil
- **Cifra de Vigenère** · Fácil
- **Impressora de etiquetas** · Médio
- **Campos de CSV** · Médio
- **Quebra de linha** · Difícil
- Teste: **Relatório de estoque** · Difícil